# Gain reference comparison
Compare the RELION-estimated gain (`estimated_gain.mrc`) with the EBIC gain reference (`gain_20260213T101027.mrc`).

Both are 5760 × 4092 px (K3 sensor). Axes use equal pixel spacing to preserve the rectangular sensor aspect ratio.

**Run from your RELION project directory** (the one containing `estimated_gain.mrc`).

In [ ]:
import mrcfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import os

# Set this to your RELION project directory
PROJECT_DIR = '/mnt/McQueen-002/sconnell/TEST-ARETOMO-PARSE/relion'
os.chdir(PROJECT_DIR)

GAIN_RELION = Path('estimated_gain.mrc')
GAIN_EBIC   = Path('gain_20260213T101027.mrc')
OUT         = Path('run001-cmd0/analyse/gain_comparison.png')

with mrcfile.open(GAIN_RELION, permissive=True) as m:
    gain_relion = m.data.copy()
with mrcfile.open(GAIN_EBIC, permissive=True) as m:
    gain_ebic = m.data.copy()

print(f'RELION  : {gain_relion.shape}  min={gain_relion.min():.3f}  max={gain_relion.max():.3f}  mean={gain_relion.mean():.4f}')
print(f'EBIC    : {gain_ebic.shape}  min={gain_ebic.min():.3f}  max={gain_ebic.max():.3f}  mean={gain_ebic.mean():.4f}')

In [ ]:
h, w = gain_relion.shape  # 4092, 5760
aspect = w / h            # ~1.407  — true K3 sensor aspect ratio

panel_h_in = 10 / 2.54
panel_w_in = panel_h_in * aspect

fig, axes = plt.subplots(1, 2, figsize=(2 * panel_w_in, panel_h_in))

gains  = [gain_relion, gain_ebic]
titles = ['RELION estimated gain', 'EBIC gain reference']

for ax, data, title in zip(axes, gains, titles):
    vmin, vmax = np.percentile(data, [1, 99])
    ax.imshow(data, cmap='gray', vmin=vmin, vmax=vmax,
              aspect='equal', interpolation='nearest')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(f'{w} px', fontsize=8)
    ax.set_ylabel(f'{h} px', fontsize=8)
    ax.tick_params(labelsize=7)

fig.tight_layout()
fig.savefig(OUT, dpi=150)
plt.show()
print(f'Saved: {OUT}')